# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratyush457/week-1-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row is one content item for one client on one report date.

**Time window:** I will use the March 2026 partition (`month=2026-03`) for development and verification. The final month, June 2026, will be treated as a sealed test window and will not be used to develop label logic.

For my refresh-opportunity lane, the decision is made at the content-item level, but the warehouse records daily performance for each client-content pair. I will aggregate or select the appropriate daily signals only after verifying the warehouse grain and date coverage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Features:** gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, scroll_events.

**Label / proxy:** A future refresh-opportunity signal based on later performance. For the leakage demonstration, future April clicks are deliberately used as a label-derived signal, but they must not be available as a feature at the March decision point.

**Context:** client_hash_id, content_hash_id, report_date, month, gsc_data_available, ga4_data_available.

**Excluded:** Future-period performance fields when making the decision, because they would leak information from the outcome window into the features. Client and content identifiers are also excluded from model features because they identify the entity rather than describing its performance.

The feature set is intentionally small. Every feature should be available at the moment the prioritization decision is made.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

march = f"""
read_parquet(
    '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Verification 1: prove the warehouse grain
grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {march}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate rows at report_date × client × content grain:")
print(grain_check)

if grain_check.empty:
    print("Grain check passed: one row = one client × content × report_date.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows at report_date × client × content grain:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []
Grain check passed: one row = one client × content × report_date.


In [2]:
# Verification 2: March 2026 row count and date span

count_span = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {march}
""").df()

print(count_span)

   row_count first_date  last_date
0    9841378 2026-03-01 2026-03-31


In [4]:
# Verification 3: GSC availability check using IS TRUE

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM {march}
""").df()

print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378         3611061


In [9]:
# Five-feature frame for March 2026
# One row = one client × content × report_date

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events
    FROM {march}
    WHERE gsc_data_available IS TRUE
    LIMIT 10
""").df()

feature_frame

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,<NA>


### Five features and when they are knowable

- **gsc_impressions:** Knowable at the decision moment because these impressions are available up to the reporting date.
- **gsc_clicks:** Knowable at the decision moment because these Search Console clicks are available up to the reporting date.
- **gsc_avg_position:** Knowable at the decision moment because the average search position is calculated from Search Console data available up to the reporting date.
- **ga4_sessions:** Knowable at the decision moment when GA4 data is available, because sessions are observed up to the reporting date.
- **scroll_events:** Knowable at the decision moment when analytics data is available, because scroll events are observed up to the reporting date.

In [7]:
# Deliberate leakage experiment — RAM-safe
# April clicks are future information for a March decision.

april = f"""
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

leak_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE a.future_clicks IS NOT NULL
        ) AS rows_with_future_info,
        AVG(a.future_clicks) AS avg_future_clicks
    FROM {march} m
    LEFT JOIN (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS future_clicks
        FROM {april}
        GROUP BY client_hash_id, content_hash_id
    ) a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
    WHERE m.gsc_data_available IS TRUE
""").df()

leak_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,rows_with_future_info,avg_future_clicks
0,3611061,3611060,6.264653


In [8]:
# Deliberate leakage:
# future_clicks_leaky is information that would NOT be known
# when making the March decision.

leaky_sample = con.sql(f"""
    SELECT
        m.gsc_impressions,
        m.gsc_clicks,
        a.future_clicks,
        CASE WHEN a.future_clicks > 0 THEN 1 ELSE 0 END AS target
    FROM {march} m
    INNER JOIN (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS future_clicks
        FROM {april}
        GROUP BY client_hash_id, content_hash_id
    ) a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id
    WHERE m.gsc_data_available IS TRUE
      AND a.future_clicks IS NOT NULL
    USING SAMPLE 50000
""").df()

print("Sample rows:", len(leaky_sample))
print("Positive targets:", leaky_sample["target"].sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample rows: 24091
Positive targets: 10825


In [9]:
# Leaky score: future information is used directly as the ranking signal

leaky_rank = leaky_sample.sort_values(
    "future_clicks",
    ascending=False
)

top_50 = leaky_rank.head(50)

leaky_precision_at_50 = top_50["target"].mean()

print(f"LEAKY Precision@50: {leaky_precision_at_50:.3f}")

LEAKY Precision@50: 1.000


### Leakage lesson

The leaky Precision@50 was **1.000** because the ranking used `future_clicks`, which comes from April 2026. April information would not be available when making the March decision.

This is data leakage: information from the outcome window was allowed into the decision-time features. The 1.000 score therefore cannot be used as evidence of real model quality.

For the honest version, future-period fields such as `future_clicks` must be removed. Only information available on or before the March reporting date should be used.

## 4. Data limits

One important limitation is **availability bias**: only 3,611,061 of the 9,841,378 March rows had `gsc_data_available IS TRUE`.

This means the GSC-based feature analysis does not represent every warehouse row equally. Pages or clients without available GSC data may be systematically different from those with data, so the resulting prioritization should not automatically be treated as representative of the entire dataset.

Another limitation is that this is observational data. The analysis can identify useful signals and support prioritization, but it cannot prove that changing a page will cause better future performance.

## 5. Self-check

- [x] I defined the warehouse grain in plain words.
- [x] I named the table/slice and used a mid-panel month for development.
- [x] I showed exactly three verification checks with visible outputs.
- [x] The availability check uses `IS TRUE`.
- [x] I built a feature frame with no more than five features.
- [x] I explained why each feature is knowable at the decision moment.
- [x] I deliberately demonstrated a leakage trap using future information.
- [x] I recorded why the leaky score cannot be trusted.
- [x] I named a limitation of the data slice.
- [x] I kept future-period information out of the honest feature set.